# DLL Lab — 05 February 2026
## Multi-Layer Perceptron — MNIST Digit Classification
**Done By:** Student | **Course:** AI-612


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow import keras
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix
import warnings; warnings.filterwarnings('ignore')
print('Libraries loaded.')


Libraries loaded.


## Load MNIST

In [2]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
X_test  = X_test.reshape(-1, 784).astype('float32')  / 255.0
y_train_oh = to_categorical(y_train, 10)
y_test_oh  = to_categorical(y_test, 10)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')


Train: (60000, 784)  Test: (10000, 784)


## Sample Images

In [3]:
fig, axes = plt.subplots(2, 5, figsize=(11, 4))
for i, ax in enumerate(axes.flatten()):
    idx = np.where(y_train == i)[0][0]
    ax.imshow(X_train[idx].reshape(28,28), cmap='gray')
    ax.set_title(f'Digit {i}'); ax.axis('off')
plt.suptitle('MNIST Sample Images', fontsize=12)
plt.tight_layout(); plt.show()


## Build MLP

In [4]:
model = Sequential([
    Dense(512, activation='relu', input_shape=(784,)),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.2),
    Dense(10,  activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 512)               401,920   
 dropout (Dropout)           (None, 512)               0         
 dense_1 (Dense)             (None, 256)               131,328   
 dropout_1 (Dropout)         (None, 256)               0         
 dense_2 (Dense)             (None, 10)                2,570     
Total params: 535,818
_________________________________________________________________


## Train

In [5]:
history = model.fit(X_train, y_train_oh, epochs=15, batch_size=128,
                    validation_data=(X_test, y_test_oh), verbose=1)


Epoch  1/15 — loss: 0.2612 — accuracy: 0.9242 — val_accuracy: 0.9712
Epoch  5/15 — loss: 0.0834 — accuracy: 0.9748 — val_accuracy: 0.9801
Epoch 10/15 — loss: 0.0523 — accuracy: 0.9842 — val_accuracy: 0.9831
Epoch 15/15 — loss: 0.0389 — accuracy: 0.9878 — val_accuracy: 0.9847


In [6]:
_, test_acc = model.evaluate(X_test, y_test_oh, verbose=0)
print(f'Test Accuracy: {test_acc:.4f}')

y_pred = np.argmax(model.predict(X_test), axis=1)
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('Confusion Matrix')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
plt.tight_layout(); plt.show()


Test Accuracy: 0.9847


## Summary
| | Value |
|---|---|
| Dataset | MNIST (60k train / 10k test) |
| Architecture | 784 → 512 → 256 → 10 |
| Test Accuracy | **98.47%** |
| Epochs | 15 |
